In [1]:
import pandas as pd
import numpy as np

print("Environment ready")

Environment ready


In [2]:
import pandas as pd

url = "https://raw.githubusercontent.com/selva86/datasets/master/BostonHousing.csv"
df = pd.read_csv(url)

df.head()

,crim,zn,indus,chas,nox,rm,age,dis,rad,tax,ptratio,b,lstat,medv
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222,18.7,396.90,5.33,36.2


In [3]:
import pandas as pd
import numpy as np

np.random.seed(42)

n = 1000

df = pd.DataFrame({
    "age": np.random.randint(20, 80, n),
    "smoker": np.random.binomial(1, 0.3, n),
})

# create hypertension with dependency on smoking + age
df["hypertension"] = (
    0.03 * df["age"] + 
    0.8 * df["smoker"] + 
    np.random.normal(0, 1, n)
)

df["hypertension"] = (df["hypertension"] > 2.5).astype(int)

df.head()

,age,smoker,hypertension
0,58,0,0
1,71,0,0
2,48,0,0
3,34,0,0
4,62,0,0


In [4]:
import statsmodels.api as sm

# define variables
X = df[["smoker", "age"]]
X = sm.add_constant(X)

y = df["hypertension"]

# fit model
model = sm.Logit(y, X).fit()

print(model.summary())

Optimization terminated successfully.
         Current function value: 0.466458
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:           hypertension   No. Observations:                 1000
Model:                          Logit   Df Residuals:                      997
Method:                           MLE   Df Model:                            2
Date:                Mon, 27 Apr 2026   Pseudo R-squ.:                  0.1830
Time:                        20:08:51   Log-Likelihood:                -466.46
converged:                       True   LL-Null:                       -570.95
Covariance Type:            nonrobust   LLR p-value:                 4.148e-46
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -4.7047      0.339    -13.859      0.000      -5.370      -4.039
smoker         1.4798      0.

## Conclusion

In this project, we analyzed the relationship between smoking and hypertension using an observational dataset.

Using logistic regression, we found that smoking and age are both significantly associated with increased risk of hypertension.

This project demonstrates:
- Application of statistical modeling (logistic regression)
- Handling of observational data
- Interpretation of results in a real-world context

Future improvements could include:
- More advanced causal inference methods (e.g., propensity scores)
- Use of real clinical datasets
- Inclusion of additional confounding variables

In [6]:
from sklearn.linear_model import LogisticRegression

# define covariates (confounders)
X_ps = df[["age"]]
y_ps = df["smoker"]

# fit propensity score model
ps_model = LogisticRegression()
ps_model.fit(X_ps, y_ps)

# get propensity scores
df["ps"] = ps_model.predict_proba(X_ps)[:, 1]

df.head()

,age,smoker,hypertension,ps
0,58,0,0,0.306737
1,71,0,0,0.304651
2,48,0,0,0.308348
3,34,0,0,0.310610
4,62,0,0,0.306095


In [7]:
# compute inverse probability weights
df["weight"] = df["smoker"]/df["ps"] + (1 - df["smoker"])/(1 - df["ps"])

df[["smoker", "ps", "weight"]].head()

,smoker,ps,weight
0,0,0.306737,1.442455
1,0,0.304651,1.438127
2,0,0.308348,1.445813
3,0,0.310610,1.450558
4,0,0.306095,1.441118


In [8]:
import statsmodels.api as sm

# define variables
X_w = df[["smoker", "age"]]
X_w = sm.add_constant(X_w)
y_w = df["hypertension"]

# weighted model
model_w = sm.GLM(y_w, X_w, 
                 family=sm.families.Binomial(), 
                 freq_weights=df["weight"]).fit()

print(model_w.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:           hypertension   No. Observations:                 1000
Model:                            GLM   Df Residuals:                  1997.00
Model Family:                Binomial   Df Model:                            2
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -997.94
Date:                Mon, 27 Apr 2026   Deviance:                       1995.9
Time:                        20:10:05   Pearson chi2:                 1.99e+03
No. Iterations:                     5   Pseudo R-squ. (CS):             0.3754
Covariance Type:            nonrobust                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -4.7157      0.234    -20.163      0.0